# BoardCheck: Week 3 V1 data pipeline (Topic 3 lab)

Builds the document pipeline for BoardCheck, the manual-grounded assistant for AM5 B650 slot and lane questions. Pathway A (PDF/TXT) only, run on all 9 real sources from `data/sources.md`: 3 board manuals (S01–S03) and 6 AMD spec pages (S04–S09).

**Requirements served:** FR1 and FR2 (slot rules and sharing footnotes live in the manuals' spec tables), FR4 (AMD pages give each CPU's usable lanes), FR5 + RAI1 (every record keeps its document and page, or spec field for AMD), FR7 + RAI3 (each source records its revision and CPU coverage), RAI4 (comparison metrics are reported per manual), RAI5 (no extracted text is published; outputs below show at most 25 words), NFR5 (re-runnable, fingerprinted pipeline).

**Changes from the handout** (everything else is the handout's code):

| Cell | Handout | This notebook | Why |
|---|---|---|---|
| 1 | Fixed `/content` path | Repository root locally, `/content` in Colab, overridable with `WEEK3_BASE_DIR`; installs missing packages only | Runs in the project repository, so outputs land in `data/` |
| 1B | none | `SOURCES` registry: the only place sources are named | One source list for Parts A, B and C |
| 2 | Extract the dummy ZIP | Download the 9 sources; upload fallback | Real project data, no dummy pack |
| 3 | Hard-codes `/content` | Uses Cell 1 paths; checks all 9 sources and the manuals' byte sizes | A replaced manual would silently move the cited pages |
| A3 | No code | Per-source inspection summary and representative records | A3 asks for inspection evidence |
| 5A1 | Whole-document text, `page: None` | Page-aware chunker, stronger cleaning, AMD field loader; handout chunkers kept as the B3 baseline | FR5/RAI1 need a page on every record |
| 5A2 | One `prepared_text.jsonl` | One `prepared_text_S0N.jsonl` per source | 1:1 source-to-output lineage |
| B3 | Existence and size check | Same check plus record-level validation and probes | A non-empty file does not prove the pages are right |
| 6A | `source_id` by position, rights "Not verified" | IDs, revision, rights and coverage from the registry | FR7/RAI3/RAI5 |
| 7 | Redefines `BASE_DIR` | Reuses Cell 1 paths; accepts a folder of documents | Local runs; 9 inputs |
| 8A | One file, handout chunkers | Calls the same Part B functions on all 9 sources | Part B and Part C cannot drift |
| 9 | Re-run check | Plus: Part C outputs byte-identical to Part B | Proves both parts used one method |
| 7, all | Prints and stores full paths | Paths relative to the project folder in outputs and manifests | Keeps local folder names out of committed files |

**Copyright (RAI5):** the manuals and AMD pages are copyrighted. Raw files and the `.jsonl` outputs stay local; the repository keeps this notebook, the manifests (hashes, counts, paths) and the docs.

*AI use: drafted with help from Claude (Anthropic). Every cell was run, checked and explained by me before submission.*

## Part A: Start with the data

In [1]:
# Cell 1 — Workspace setup (RUN FIRST)
# Change from handout: BASE_DIR is the repository root when the notebook runs
# locally from notebooks/, and /content/week3_project in Colab. The
# WEEK3_BASE_DIR environment variable overrides both (used for test runs).
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DEFAULT_BASE_DIR = Path("/content/week3_project") if IN_COLAB else Path.cwd().parent
BASE_DIR = Path(os.environ.get("WEEK3_BASE_DIR", DEFAULT_BASE_DIR)).resolve()
if not IN_COLAB and "WEEK3_BASE_DIR" not in os.environ:
    assert (BASE_DIR / "data" / "sources.md").exists(), (
        f"Expected the repository root, got {BASE_DIR}. Run this notebook from notebooks/."
    )
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases for notebook examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

# Install only what is missing (module name -> pip package)
REQUIRED_PACKAGES = {"pypdf": "pypdf", "fontTools": "fonttools", "bs4": "beautifulsoup4",
                     "pandas": "pandas"}
missing = [pkg for module, pkg in REQUIRED_PACKAGES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import bs4
import pypdf

print("Project folder:", BASE_DIR.name, "| Colab" if IN_COLAB else "| local")
print("Raw folder:", RAW_DIR.relative_to(BASE_DIR))
print("Processed folder:", PROCESSED_DIR.relative_to(BASE_DIR))
print("pypdf", pypdf.__version__, "| beautifulsoup4", bs4.__version__)

Project folder: boardcheck | local
Raw folder: data/raw
Processed folder: data/processed
pypdf 7.1.0 | beautifulsoup4 4.15.0


In [2]:
# Cell 1B — Source registry (added)
# The only place sources are named. Order, URLs and sizes follow data/sources.md.
AMD_BASE_URL = "https://www.amd.com/en/products/processors/desktops/ryzen"
AMD_RIGHTS = "AMD website terms: personal, non-public use. Local copy only, not redistributed."
AMD_REVISION = "Web page with no revision number; retrieval time recorded in the manifest"


def amd_source(source_id, series, slug, title):
    return {
        "source_id": source_id,
        "relative_path": f"amd/{slug}.html",
        "url": f"{AMD_BASE_URL}/{series}/{slug}.html",
        "expected_bytes": None,
        "doc_title": title,
        "revision": AMD_REVISION,
        "cpu_coverage": "n/a (CPU spec page)",
        "locator_type": "spec_field",
        "page_range": None,
        "rights": AMD_RIGHTS,
    }


SOURCES = [
    {
        "source_id": "S01",
        "relative_path": "manuals/MAGB650TOMAHAWKWIFI.pdf",
        "url": "https://download.msi.com/archive/mnu_exe/mb/MAGB650TOMAHAWKWIFI.pdf",
        "expected_bytes": 37_426_776,
        "doc_title": "MSI MAG B650 TOMAHAWK WIFI user guide",
        "revision": "No edition code printed; PDF file date 2024-12-18",
        "cpu_coverage": "Ryzen 9000/8000/7000",
        "locator_type": "pdf_page",
        "page_range": [5, 57],   # English section without contents pages; p.58 starts German
        "rights": "MSI copyrighted manual. Local copy only, not redistributed.",
    },
    {
        "source_id": "S02",
        "relative_path": "manuals/mb_manual_b650-aorus-elite-ax_1204_e.pdf",
        "url": "https://download.gigabyte.com/FileList/Manual/mb_manual_b650-aorus-elite-ax_1204_e.pdf",
        "expected_bytes": 17_239_437,
        "doc_title": "Gigabyte B650 AORUS ELITE AX user's manual",
        "revision": "Rev 1204; PDF file date 2024-12-06",
        "cpu_coverage": "Ryzen 9000/8000/7000",
        "locator_type": "pdf_page",
        "page_range": [4, 36],   # pp.37-41 are conformity declarations and contacts
        "rights": "Gigabyte forbids reproduction without written permission. Local copy only.",
    },
    {
        "source_id": "S03",
        "relative_path": "manuals/E20268_ROG_STRIX_B650-A_GAMING_WIFI_UM_WEB.pdf",
        "url": "https://dlcdnets.asus.com/pub/ASUS/mb/Socket%20AM5/ROG%20STRIX%20B650-A%20GAMING%20WIFI/E20268_ROG_STRIX_B650-A_GAMING_WIFI_UM_WEB.pdf",
        "expected_bytes": 6_386_249,
        "doc_title": "ASUS ROG STRIX B650-A GAMING WIFI user manual",
        "revision": "E20268 First Edition, September 2022",
        "cpu_coverage": "Ryzen 7000 only (no edition covers Ryzen 8000/9000)",
        "locator_type": "pdf_page",
        "page_range": [4, 66],   # pp.67-80 are regulatory notices and warranty
        "rights": ("ASUS copyrighted; PDF permissions disallow copying (copy:no). "
                   "Text extracted for private coursework only, not redistributed."),
    },
    amd_source("S04", "9000-series", "amd-ryzen-7-9700x", "AMD Ryzen 7 9700X spec page"),
    amd_source("S05", "7000-series", "amd-ryzen-5-7600", "AMD Ryzen 5 7600 spec page"),
    amd_source("S06", "8000-series", "amd-ryzen-7-8700g", "AMD Ryzen 7 8700G spec page"),
    amd_source("S07", "8000-series", "amd-ryzen-5-8600g", "AMD Ryzen 5 8600G spec page"),
    amd_source("S08", "8000-series", "amd-ryzen-5-8500g", "AMD Ryzen 5 8500G spec page"),
    amd_source("S09", "8000-series", "amd-ryzen-3-8300g", "AMD Ryzen 3 8300G spec page"),
]
SOURCE_BY_ID = {source["source_id"]: source for source in SOURCES}

print("Registered sources:", len(SOURCES))
for source in SOURCES:
    print(f"  {source['source_id']}  {source['locator_type']:<10}  {source['relative_path']}")

Registered sources: 9
  S01  pdf_page    manuals/MAGB650TOMAHAWKWIFI.pdf
  S02  pdf_page    manuals/mb_manual_b650-aorus-elite-ax_1204_e.pdf
  S03  pdf_page    manuals/E20268_ROG_STRIX_B650-A_GAMING_WIFI_UM_WEB.pdf
  S04  spec_field  amd/amd-ryzen-7-9700x.html
  S05  spec_field  amd/amd-ryzen-5-7600.html
  S06  spec_field  amd/amd-ryzen-7-8700g.html
  S07  spec_field  amd/amd-ryzen-5-8600g.html
  S08  spec_field  amd/amd-ryzen-5-8500g.html
  S09  spec_field  amd/amd-ryzen-3-8300g.html


In [3]:
# Cell 2 — Ingest sources (replaces the optional dummy-ZIP cell)
# Downloads any registered source that is not already in data/raw.
# amd.com refuses Python's default client, so requests send browser headers.
# If a download fails, upload that file with the Files panel into the same
# sub-folder of data/raw and rerun this cell.
import urllib.request

BROWSER_HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64; rv:140.0) Gecko/20100101 Firefox/140.0",
    "Accept": "text/html,application/xhtml+xml,application/pdf;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}
DOWNLOAD_TIMEOUT_S = 120


def download(url, target):
    request = urllib.request.Request(url, headers=BROWSER_HEADERS)
    with urllib.request.urlopen(request, timeout=DOWNLOAD_TIMEOUT_S) as response:
        data = response.read()
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(data)


failed = []
for source in SOURCES:
    target = RAW_DIR / source["relative_path"]
    if target.is_file():
        print("Present:   ", source["relative_path"])
        continue
    try:
        download(source["url"], target)
        print("Downloaded:", source["relative_path"])
    except OSError as error:
        failed.append(source["relative_path"])
        print("FAILED:    ", source["relative_path"], "->", error)

if failed:
    raise RuntimeError(
        "Upload these files into data/raw/ (same sub-folders), then rerun Cell 2: "
        + ", ".join(failed)
    )
print("Cell 2 completed: all sources are in data/raw.")

Present:    manuals/MAGB650TOMAHAWKWIFI.pdf
Present:    manuals/mb_manual_b650-aorus-elite-ax_1204_e.pdf
Present:    manuals/E20268_ROG_STRIX_B650-A_GAMING_WIFI_UM_WEB.pdf
Present:    amd/amd-ryzen-7-9700x.html
Present:    amd/amd-ryzen-5-7600.html
Present:    amd/amd-ryzen-7-8700g.html
Present:    amd/amd-ryzen-5-8600g.html
Present:    amd/amd-ryzen-5-8500g.html
Present:    amd/amd-ryzen-3-8300g.html
Cell 2 completed: all sources are in data/raw.


In [4]:
# Cell 3 — Verify uploaded inputs
# Changes from handout: paths come from Cell 1 instead of a hard-coded
# /content path, and every registered source must be present. Manual byte
# sizes must match data/sources.md: a different size means the vendor
# replaced the file, so the gold-set pages must be re-checked before use.
files = sorted(f for f in RAW_DIR.rglob("*") if f.is_file())

print("Input count:", len(files))
for f in files:
    print(f.relative_to(RAW_DIR), "bytes:", f.stat().st_size)

assert files, "Upload at least one relevant input file before proceeding."

for source in SOURCES:
    path = RAW_DIR / source["relative_path"]
    assert path.is_file(), f"Missing {source['source_id']}: {path}"
    if source["expected_bytes"] is not None:
        assert path.stat().st_size == source["expected_bytes"], (
            f"{source['source_id']} is {path.stat().st_size} bytes, expected "
            f"{source['expected_bytes']}: the vendor may have replaced the file."
        )

print("Upload verified: all", len(SOURCES), "registered sources present; manual sizes match.")

Input count: 9
amd/amd-ryzen-3-8300g.html bytes: 202315
amd/amd-ryzen-5-7600.html bytes: 194985
amd/amd-ryzen-5-8500g.html bytes: 202442
amd/amd-ryzen-5-8600g.html bytes: 202717
amd/amd-ryzen-7-8700g.html bytes: 204067
amd/amd-ryzen-7-9700x.html bytes: 204719
manuals/E20268_ROG_STRIX_B650-A_GAMING_WIFI_UM_WEB.pdf bytes: 6386249
manuals/MAGB650TOMAHAWKWIFI.pdf bytes: 37426776
manuals/mb_manual_b650-aorus-elite-ax_1204_e.pdf bytes: 17239437
Upload verified: all 9 registered sources present; manual sizes match.


### A3: Inspect the selected sources

Pathway A covers PDF and TXT. The AMD pages are HTML, so they get their own loader (BeautifulSoup reads the `<dt>`/`<dd>` spec list). Text is only shown through `preview()`, capped at 25 words.

In [5]:
# Cell A3 — Source inspection (added; the handout gives no code for A3)
import re
import statistics

from bs4 import BeautifulSoup
from pypdf import PdfReader

PREVIEW_WORDS = 25
NEAR_EMPTY_CHARS = 40
BOX_GLYPH = "□"
CONTROL_CHARS = re.compile(r"[\x00-\x08\x0b-\x1f\x7f-\x9f]")


def normalise(text):
    """Collapse every run of whitespace (spaces, tabs, newlines) to one space."""
    return " ".join(text.split())


def preview(text, words=PREVIEW_WORDS):
    """Short excerpt only: the sources are copyrighted (RAI5)."""
    tokens = text.split()
    return " ".join(tokens[:words]) + (" ..." if len(tokens) > words else "")


def open_pdf(path):
    reader = PdfReader(str(path))
    if reader.is_encrypted:
        reader.decrypt("")   # S03 is RC4-encrypted with an empty user password
    return reader


def inspect_pdf(source):
    path = RAW_DIR / source["relative_path"]
    reader = open_pdf(path)
    first, last = source["page_range"]
    texts = {n: reader.pages[n - 1].extract_text() or "" for n in range(first, last + 1)}
    lengths = {n: len(t.strip()) for n, t in texts.items()}
    return {
        "source_id": source["source_id"],
        "pdf_pages": len(reader.pages),
        "encrypted": reader.is_encrypted,
        "page_range": f"{first}-{last}",
        "chars_min": min(lengths.values()),
        "chars_median": int(statistics.median(lengths.values())),
        "chars_max": max(lengths.values()),
        "near_empty_pages": [n for n, size in lengths.items() if size < NEAR_EMPTY_CHARS],
        "box_glyphs": sum(t.count(BOX_GLYPH) for t in texts.values()),
        "control_chars": sum(len(CONTROL_CHARS.findall(t)) for t in texts.values()),
    }


def amd_spec_fields(path):
    """One dict per <dt>/<dd> pair, with the heading of the section it sits under."""
    soup = BeautifulSoup(path.read_text(encoding="utf-8"), "html.parser")
    fields = []
    for dt in soup.find_all("dt"):
        dd = dt.find_next_sibling("dd")
        if dd is None:
            raise ValueError(f"<dt> without <dd> in {path.name}: {dt.get_text(strip=True)}")
        heading = dt.find_previous(["h2", "h3", "h4"])
        fields.append({
            "section": normalise(heading.get_text(" ")) if heading else None,
            "field": normalise(dt.get_text(" ")),
            "value": normalise(dd.get_text(" ")),
        })
    if not fields:
        raise ValueError(f"No spec fields in {path.name}: the page layout may have changed.")
    return fields


def inspect_amd(source):
    fields = amd_spec_fields(RAW_DIR / source["relative_path"])
    return {
        "source_id": source["source_id"],
        "spec_fields": len(fields),
        "sections": len({f["section"] for f in fields}),
        "fields_without_section": sum(f["section"] is None for f in fields),
        "empty_values": sum(not f["value"] for f in fields),
    }


print("--- Manuals ---")
for source in SOURCES:
    if source["locator_type"] == "pdf_page":
        print(inspect_pdf(source))
print("\n--- AMD spec pages ---")
for source in SOURCES:
    if source["locator_type"] == "spec_field":
        print(inspect_amd(source))

# Representative records: the slot rows behind the Week 2 gold set
print("\n--- Representative content (25-word previews) ---")
for source_id, page in (("S01", 18), ("S02", 9), ("S03", 6)):
    reader = open_pdf(RAW_DIR / SOURCE_BY_ID[source_id]["relative_path"])
    text = reader.pages[page - 1].extract_text() or ""
    print(f"{source_id} p.{page}: {preview(text)}")
for field in amd_spec_fields(RAW_DIR / SOURCE_BY_ID["S08"]["relative_path"]):
    if "Lanes" in field["field"]:
        print(f"S08 {field['section']} > {field['field']}: {field['value']}")

--- Manuals ---
{'source_id': 'S01', 'pdf_pages': 453, 'encrypted': False, 'page_range': '5-57', 'chars_min': 23, 'chars_median': 556, 'chars_max': 2284, 'near_empty_pages': [16, 17], 'box_glyphs': 0, 'control_chars': 0}
{'source_id': 'S02', 'pdf_pages': 41, 'encrypted': False, 'page_range': '4-36', 'chars_min': 378, 'chars_median': 1579, 'chars_max': 4795, 'near_empty_pages': [], 'box_glyphs': 6138, 'control_chars': 90}
{'source_id': 'S03', 'pdf_pages': 80, 'encrypted': True, 'page_range': '4-66', 'chars_min': 3, 'chars_median': 718, 'chars_max': 2291, 'near_empty_pages': [12], 'box_glyphs': 0, 'control_chars': 0}

--- AMD spec pages ---
{'source_id': 'S04', 'spec_fields': 54, 'sections': 5, 'fields_without_section': 0, 'empty_values': 0}
{'source_id': 'S05', 'spec_fields': 57, 'sections': 5, 'fields_without_section': 0, 'empty_values': 0}
{'source_id': 'S06', 'spec_fields': 67, 'sections': 6, 'fields_without_section': 0, 'empty_values': 0}
{'source_id': 'S07', 'spec_fields': 66, 'sec

In [6]:
# Cell 4 — REQUIRED: Source Selection + Lineage Utilities (handout, unchanged)
# Run ONCE after Cell 1 and Cell 3
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone


# Select the exact source before parsing.
# Never automatically take the first file by extension.
def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(
                f"Source not found: {selected}"
            )
        return selected

    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )

    print(
        "Candidates:",
        [str(p.relative_to(RAW_DIR)) for p in matches]
    )

    if len(matches) != 1:
        raise ValueError(
            "Select exactly one source. "
            "Set relative_path to an exact file under RAW_DIR."
        )

    return matches[0]


def source_hash(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }

    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)

    target.write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print("Saved lineage:", target.relative_to(BASE_DIR))
    return manifest


# Confirmation messages
print("Cell 4 completed successfully.")
print("Source selection and lineage utilities are ready.")
print("Available functions:")
print("- select_source()")
print("- source_hash()")
print("- save_manifest()")

Cell 4 completed successfully.
Source selection and lineage utilities are ready.
Available functions:
- select_source()
- source_hash()
- save_manifest()


In [7]:
# CELL 4A — COMMON LINEAGE REGISTRY (handout, unchanged)
# REQUIRED: Run once before selected Part B pathways
from pathlib import Path

lineage_entries = {}


def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()

    # Validate source
    if not source.is_file():
        raise FileNotFoundError(
            f"Source file not found: {source}"
        )

    # Validate prepared output
    if not output.is_file():
        raise FileNotFoundError(
            f"Prepared output not found: {output}"
        )

    if output.stat().st_size == 0:
        raise ValueError(
            f"Prepared output is empty: {output}"
        )

    # Validate processing description
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError(
            "A processing description is required."
        )

    processing = processing.strip()

    if processing.upper().startswith("REPLACE"):
        raise ValueError(
            "Replace the placeholder with the actual method."
        )

    # Ensure source and output belong to project workspace
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())

    # Same source/output pair is updated, not duplicated
    key = (str(source), str(output))

    lineage_entries[key] = {
        "source": source,
        "output": output,
        "processing": processing
    }

    print(
        "Registered:",
        source.name,
        "->",
        output.name
    )


print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))

Common lineage registry ready.
Registered entries: 0


## Part B: Prepare the data

The handout's chunkers join every page into one string, so each record has `page: None`; the handout notes say to "develop a page-aware splitter" when page provenance matters. BoardCheck must cite a page for every claim (FR5, RAI1), so the selected method chunks each page on its own.

In [8]:
# Cell 5A1 (part 1) — Extraction and cleaning
# Cleaning = the handout's space/tab rule plus three fixes found in Phase 1:
#   * Gigabyte uses control characters as bullet glyphs (pp. 8-15)
#   * Gigabyte diagram labels extract as runs of "□" (6,138 on pp. 23-30)
#   * dot leaders ("....") from contents-style lines
HANDOUT_WHITESPACE = re.compile(r"[ \t]+")
BOX_GLYPH_TOKEN = re.compile(r"\S*□\S*")
DOT_LEADER = re.compile(r"\.{4,}")


def clean_page(text):
    text = CONTROL_CHARS.sub(" ", text)
    text = BOX_GLYPH_TOKEN.sub(" ", text)
    text = DOT_LEADER.sub(" ", text)
    return HANDOUT_WHITESPACE.sub(" ", text).strip()


def extract_pages(source):
    """(page_no, cleaned text) for each page in the source's range. page_no = 1-based PDF index."""
    reader = open_pdf(RAW_DIR / source["relative_path"])
    first, last = source["page_range"]
    return [(n, clean_page(reader.pages[n - 1].extract_text() or ""))
            for n in range(first, last + 1)]


PAGES = {s["source_id"]: extract_pages(s) for s in SOURCES if s["locator_type"] == "pdf_page"}
for source_id, pages in PAGES.items():
    raw_reader = open_pdf(RAW_DIR / SOURCE_BY_ID[source_id]["relative_path"])
    raw_chars = sum(len(raw_reader.pages[n - 1].extract_text() or "") for n, _ in pages)
    clean_chars = sum(len(text) for _, text in pages)
    print(f"{source_id}: {len(pages)} pages, {raw_chars} raw chars -> {clean_chars} cleaned")

S01: 53 pages, 35324 raw chars -> 35285 cleaned
S02: 33 pages, 55211 raw chars -> 48008 cleaned
S03: 63 pages, 50885 raw chars -> 50845 cleaned


In [9]:
# Cell 5A1 (part 2) — Chunkers


# Handout baselines, unchanged (Cell 5A1 steps 4-5)
def fixed_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    return [
        s[i:i + size]
        for i in range(0, len(s), size)
    ]


def paragraph_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    parts = []
    current = ""
    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()
        if not segment:
            continue
        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""
            parts.extend(fixed_chunks(segment, size))
        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment
        else:
            if current:
                parts.append(current)
            current = segment
    if current:
        parts.append(current)
    return parts


# Selected method: page-aware line packing
CHUNK_SIZE = 1500        # characters; keeps chunks under a 512-token embedding limit
CHUNK_OVERLAP = 150      # characters of whole trailing lines repeated in the next chunk
MIN_PAGE_LETTERS = 40    # pages with fewer letters are diagrams or blank, skipped and logged


def letter_count(text):
    return sum(ch.isalpha() for ch in text)


def split_long_line(line, size):
    return [line[i:i + size] for i in range(0, len(line), size)]


def overlap_tail(lines, overlap):
    """Trailing whole lines whose total length fits in `overlap`."""
    tail, used = [], 0
    for line in reversed(lines):
        if used + len(line) + 1 > overlap:
            break
        tail.insert(0, line)
        used += len(line) + 1
    return tail


def pack_lines(lines, size, overlap):
    """Pack whole lines into chunks of at most `size` characters."""
    chunks, current = [], []
    for line in lines:
        if current and len("\n".join(current + [line])) > size:
            chunks.append("\n".join(current))
            current = overlap_tail(current, overlap)
            if len("\n".join(current + [line])) > size:
                current = []   # overlap + next line would exceed the cap: drop the overlap
        current.append(line)
    if current:
        chunks.append("\n".join(current))
    return chunks


def page_aware_chunks(pages, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Chunks never cross a page. Returns ([(page_no, chunk_index, text)], skipped_pages)."""
    chunks, skipped = [], []
    for page_no, text in pages:
        if letter_count(text) < MIN_PAGE_LETTERS:
            skipped.append(page_no)
            continue
        lines = [piece
                 for line in text.split("\n") if line.strip()
                 for piece in split_long_line(line.strip(), size)]
        for index, chunk in enumerate(pack_lines(lines, size, overlap), 1):
            chunks.append((page_no, index, chunk))
    return chunks, skipped


print("Chunkers ready. Selected:", f"page-aware lines, {CHUNK_SIZE} chars, {CHUNK_OVERLAP} overlap")

Chunkers ready. Selected: page-aware lines, 1500 chars, 150 overlap


In [10]:
# Cell 5A1 (part 3) — B3 comparison on the same input
# Probes: slot rules already cited in Week 2. A probe passes when one chunk
# holds every term AND that chunk's page is one of the expected pages.
# PROBES must pass (regression check). KNOWN_GAP_PROBES are table rows whose
# footnote sits far down the page; they are expected to fail until ADR 2
# (row + footnote units) is built, and are reported, not asserted.
import pandas as pd

PROBES = [
    ("S01", ["PCI_E1", "8500/ 8300"], {18}),
    ("S01", ["M2_2", "unavailable", "8500/ 8300"], {19}),
    ("S01", ["PCI_E2", "M2_3", "share the bandwidth"], {18, 19, 31}),
    ("S01", ["single PCIe x16 expansion card", "PCI_E1 slot is recommended"], {31}),
    ("S02", ["PCIEX16", "Phoenix 2", "x4 mode"], {9}),
    ("S02", ["M2B_CPU", "Phoenix 2", "PCIe 4.0 x2 SSDs"], {9}),
    ("S03", ["M.2_3", "PCIEx16_2", "disabled"], {6, 10}),
    ("S03", ["x4 mode", "shares bandwidth with M.2_3"], {6}),
]
KNOWN_GAP_PROBES = [
    ("S02", ["M2B_CPU a(Note)", "Actual support may vary by CPU"], {26}),
]
SLOT_TERM = re.compile(r"PCI_?E|PCIEX|M\.2_|M2_|M2[ABC]_|\bx(?:16|8|4|2)\b")
SLOT_HEAVY_MIN_HITS = 3


def probe_results(chunks, source_id, probes=PROBES):
    """chunks: [(page_or_None, text)]. Returns (terms_found, terms_found_on_right_page)."""
    terms_found = on_page = 0
    for probe_source, terms, pages in probes:
        if probe_source != source_id:
            continue
        hits = [page for page, text in chunks if all(t in normalise(text) for t in terms)]
        terms_found += bool(hits)
        on_page += any(page in pages for page in hits)
    return terms_found, on_page


def ends_mid_line(chunk, source_lines):
    last_line = chunk.rstrip("\n").split("\n")[-1].strip()
    return last_line not in source_lines


def compare(source_id, label, chunks):
    pages = PAGES[source_id]
    source_lines = {line.strip() for _, text in pages for line in text.split("\n")}
    texts = [normalise(text) for _, text in chunks]
    slot_heavy = [normalise(text) for _, text in pages
                  if len(SLOT_TERM.findall(text)) >= SLOT_HEAVY_MIN_HITS]
    n_probes = sum(p[0] == source_id for p in PROBES)
    terms_found, on_page = probe_results(chunks, source_id)
    n_gap = sum(p[0] == source_id for p in KNOWN_GAP_PROBES)
    _, gap_on_page = probe_results(chunks, source_id, KNOWN_GAP_PROBES)
    return {
        "source": source_id,
        "method": label,
        "chunks": len(chunks),
        "with_page_%": round(100 * sum(p is not None for p, _ in chunks) / len(chunks)),
        "ends_mid_line": sum(ends_mid_line(text, source_lines) for _, text in chunks[:-1]),
        "slot_pages_whole": f"{sum(any(page in t for t in texts) for page in slot_heavy)}/{len(slot_heavy)}",
        "max_chars": max(len(text) for _, text in chunks),
        "probe_terms": f"{terms_found}/{n_probes}",
        "probe_terms+page": f"{on_page}/{n_probes}",
        "known_gap_probe": f"{gap_on_page}/{n_gap}" if n_gap else "-",
    }


rows = []
for source_id, pages in PAGES.items():
    whole_text = "\n".join(text for _, text in pages)
    rows.append(compare(source_id, "handout fixed-500",
                        [(None, c) for c in fixed_chunks(whole_text, 500)]))
    rows.append(compare(source_id, "handout paragraph-500",
                        [(None, c) for c in paragraph_chunks(whole_text, 500)]))
    for size, overlap in ((500, 75), (CHUNK_SIZE, CHUNK_OVERLAP)):
        chunks, _ = page_aware_chunks(pages, size, overlap)
        rows.append(compare(source_id, f"page-aware {size}/{overlap}",
                            [(page, text) for page, _, text in chunks]))
    print(f"{source_id}: blank-line paragraphs in the cleaned text =",
          len(re.split(r"\n\s*\n", whole_text)))

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))

S01: blank-line paragraphs in the cleaned text = 1
S02: blank-line paragraphs in the cleaned text = 56
S03: blank-line paragraphs in the cleaned text = 4
source                method  chunks  with_page_%  ends_mid_line slot_pages_whole  max_chars probe_terms probe_terms+page known_gap_probe
   S01     handout fixed-500      71            0             67              0/6        500         4/4              0/4               -
   S01 handout paragraph-500      71            0             67              0/6        500         4/4              0/4               -
   S01     page-aware 500/75     103          100              0              3/6        498         4/4              4/4               -
   S01   page-aware 1500/150      55          100              0              6/6       1497         4/4              4/4               -
   S02     handout fixed-500      97            0             85              0/5        500         1/2              0/2             0/1
   S02 handout par

In [11]:
# Cell 5A2 — Export prepared records, one JSONL file per source
# Changes from handout: one file per source; every record keeps its page
# (manuals) or spec section + field (AMD pages).
import json

PDF_PROCESSING = f"page-aware-lines-{CHUNK_SIZE}-ov{CHUNK_OVERLAP}"
AMD_PROCESSING = "amd-spec-fields-dt-dd"


def base_record(source, sha256):
    return {
        "source_id": source["source_id"],
        "source": source["relative_path"],
        "source_sha256": sha256,
        "doc_title": source["doc_title"],
        "revision": source["revision"],
        "locator_type": source["locator_type"],
    }


def pdf_records(source, sha256, size, overlap):
    chunks, skipped = page_aware_chunks(extract_pages(source), size, overlap)
    records = [{
        "record_id": f"{source['source_id']}-p{page:03d}-c{index:02d}",
        **base_record(source, sha256),
        "page": page,
        "spec_section": None,
        "spec_field": None,
        "chunk_index": index,
        "n_chars": len(text),
        "processing": f"page-aware-lines-{size}-ov{overlap}",
        "text": text,
    } for page, index, text in chunks]
    return records, skipped


def amd_records(source, sha256):
    fields = amd_spec_fields(RAW_DIR / source["relative_path"])
    cpu_name = source["doc_title"].removesuffix(" spec page")
    records = []
    for index, field in enumerate(fields, 1):
        text = f"{cpu_name} | {field['section']} | {field['field']}: {field['value']}"
        records.append({
            "record_id": f"{source['source_id']}-f{index:03d}",
            **base_record(source, sha256),
            "page": None,
            "spec_section": field["section"],
            "spec_field": field["field"],
            "chunk_index": None,
            "n_chars": len(text),
            "processing": AMD_PROCESSING,
            "text": text,
        })
    return records, []


def build_records(source, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    sha256 = source_hash(RAW_DIR / source["relative_path"])
    if source["locator_type"] == "pdf_page":
        return pdf_records(source, sha256, size, overlap)
    return amd_records(source, sha256)


def write_jsonl(records, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as output:
        for row in records:
            output.write(json.dumps(row, ensure_ascii=False) + "\n")


def read_jsonl(path):
    with path.open("r", encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


PREPARED = {}
for source in SOURCES:
    records, skipped = build_records(source)
    prepared_path = PROCESSED_DIR / f"prepared_text_{source['source_id']}.jsonl"
    write_jsonl(records, prepared_path)
    PREPARED[source["source_id"]] = {"path": prepared_path, "records": records, "skipped": skipped}
    print(f"{source['source_id']}: {len(records):4d} records -> {prepared_path.name}"
          + (f"  (skipped pages {skipped})" if skipped else ""))

print("PATHWAY A2 completed:", sum(len(p["records"]) for p in PREPARED.values()), "records in",
      len(PREPARED), "files.")

S01:   55 records -> prepared_text_S01.jsonl  (skipped pages [16, 17])
S02:   47 records -> prepared_text_S02.jsonl
S03:   68 records -> prepared_text_S03.jsonl  (skipped pages [12, 36, 38, 66])
S04:   54 records -> prepared_text_S04.jsonl
S05:   57 records -> prepared_text_S05.jsonl
S06:   67 records -> prepared_text_S06.jsonl
S07:   66 records -> prepared_text_S07.jsonl
S08:   68 records -> prepared_text_S08.jsonl
S09:   69 records -> prepared_text_S09.jsonl
PATHWAY A2 completed: 551 records in 9 files.


In [12]:
# B3 — Basic output contract (handout) + record-level validation (added)
# Handout contract, run for every prepared file
for source_id, prepared in PREPARED.items():
    prepared_path = prepared["path"]
    assert prepared_path.exists(), 'Prepared output missing.'
    assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'
    print('Output:', prepared_path.relative_to(BASE_DIR), 'Size:', prepared_path.stat().st_size)
print('Validation: files exist and are non-empty; record checks follow.\n')


def validate_records(source, prepared):
    records, skipped = prepared["records"], prepared["skipped"]
    assert records, f"{source['source_id']}: no records"
    assert all(r["text"].strip() for r in records), "empty text"
    assert all(r["n_chars"] == len(r["text"]) for r in records), "n_chars mismatch"
    assert len(read_jsonl(prepared["path"])) == len(records), "JSONL round-trip count differs"
    if source["locator_type"] == "pdf_page":
        first, last = source["page_range"]
        assert all(first <= r["page"] <= last for r in records), "page outside range"
        assert all(r["n_chars"] <= CHUNK_SIZE for r in records), "chunk over size cap"
        pages_with_records = {r["page"] for r in records}
        missing = set(range(first, last + 1)) - pages_with_records - set(skipped)
        assert not missing, f"pages lost without being logged: {sorted(missing)}"
    else:
        assert all(r["page"] is None and r["spec_field"] for r in records), "AMD locator missing"


all_ids = [r["record_id"] for p in PREPARED.values() for r in p["records"]]
assert len(all_ids) == len(set(all_ids)), "duplicate record_id"
for source in SOURCES:
    validate_records(source, PREPARED[source["source_id"]])
    print("PASS", source["source_id"])

for source_id in PAGES:
    chunks = [(r["page"], r["text"]) for r in PREPARED[source_id]["records"]]
    n_probes = sum(p[0] == source_id for p in PROBES)
    _, on_page = probe_results(chunks, source_id)
    assert on_page == n_probes, f"{source_id}: regression probe failed ({on_page}/{n_probes})"
    print(f"{source_id} probes on exported records: {on_page}/{n_probes} PASS")
for source_id, terms, pages in KNOWN_GAP_PROBES:
    chunks = [(r["page"], r["text"]) for r in PREPARED[source_id]["records"]]
    _, on_page = probe_results(chunks, source_id, [(source_id, terms, pages)])
    print(f"{source_id} known-gap probe {terms} p.{sorted(pages)}:",
          "passes" if on_page else "fails as expected (row and footnote in different chunks; ADR 2)")

# Representative prepared records (previews only)
print("\n--- Representative records ---")
for record_id in ("S01-p018-c01", "S02-p009-c01", "S03-p006-c01"):
    record = next(r for p in PREPARED.values() for r in p["records"] if r["record_id"] == record_id)
    print(record_id, f"page {record['page']}, {record['n_chars']} chars:", preview(record["text"]))
for record in PREPARED["S08"]["records"]:
    if "Lanes" in record["spec_field"]:
        print(record["record_id"], record["text"])

Output: data/processed/prepared_text_S01.jsonl Size: 62800
Output: data/processed/prepared_text_S02.jsonl Size: 71967
Output: data/processed/prepared_text_S03.jsonl Size: 86517
Output: data/processed/prepared_text_S04.jsonl Size: 31949
Output: data/processed/prepared_text_S05.jsonl Size: 33430
Output: data/processed/prepared_text_S06.jsonl Size: 40112
Output: data/processed/prepared_text_S07.jsonl Size: 39492
Output: data/processed/prepared_text_S08.jsonl Size: 40712
Output: data/processed/prepared_text_S09.jsonl Size: 41264
Validation: files exist and are non-empty; record checks follow.

PASS S01
PASS S02
PASS S03
PASS S04
PASS S05
PASS S06
PASS S07
PASS S08
PASS S09
S01 probes on exported records: 4/4 PASS
S02 probes on exported records: 2/2 PASS
S03 probes on exported records: 2/2 PASS
S02 known-gap probe ['M2B_CPU a(Note)', 'Actual support may vary by CPU'] p.[26]: fails as expected (row and footnote in different chunks; ADR 2)

--- Representative records ---
S01-p018-c01 page 18,

In [13]:
# B4.1 — Register Pathway A lineage, one entry per source
for source in SOURCES:
    prepared = PREPARED[source["source_id"]]
    register_lineage(
        source=RAW_DIR / source["relative_path"],
        output=prepared["path"],
        processing=prepared["records"][0]["processing"],
    )

Registered: MAGB650TOMAHAWKWIFI.pdf -> prepared_text_S01.jsonl
Registered: mb_manual_b650-aorus-elite-ax_1204_e.pdf -> prepared_text_S02.jsonl
Registered: E20268_ROG_STRIX_B650-A_GAMING_WIFI_UM_WEB.pdf -> prepared_text_S03.jsonl
Registered: amd-ryzen-7-9700x.html -> prepared_text_S04.jsonl
Registered: amd-ryzen-5-7600.html -> prepared_text_S05.jsonl
Registered: amd-ryzen-7-8700g.html -> prepared_text_S06.jsonl
Registered: amd-ryzen-5-8600g.html -> prepared_text_S07.jsonl
Registered: amd-ryzen-5-8500g.html -> prepared_text_S08.jsonl
Registered: amd-ryzen-3-8300g.html -> prepared_text_S09.jsonl


In [14]:
# CELL 6A — GENERATE COMMON LINEAGE MANIFEST
# Changes from handout: source_id, version, access rights and coverage come
# from the registry instead of "Not verified"; extra fields record the URL,
# record count, page range, skipped pages, retrieval time and limitations.
from datetime import datetime, timezone

KNOWN_LIMITATIONS = {
    "pdf_page": [
        "A table row and its footnote can land in different chunks (ADR 2 deferred).",
        "Chunk size is counted in characters, not tokens.",
        "Diagram-only pages are skipped; diagram labels on kept pages remain as noise.",
    ],
    "spec_field": [
        "Live web page: a later download can differ, so the hash identifies this snapshot only.",
    ],
}

assert lineage_entries, (
    "No lineage entries registered. "
    "Register at least one selected pathway first."
)

lineage_records = []
for source in SOURCES:
    raw_path = (RAW_DIR / source["relative_path"]).resolve()
    prepared = PREPARED[source["source_id"]]
    entry = lineage_entries[(str(raw_path), str(prepared["path"].resolve()))]
    source_file, output = entry["source"], entry["output"]

    # Validate registered files again
    assert source_file.is_file(), f"Missing source: {source_file}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"

    lineage_records.append({
        "source_id": source["source_id"],
        "source_filename": str(source_file.relative_to(RAW_DIR.resolve())),
        "source_sha256": source_hash(source_file),
        "prepared_output": str(output.relative_to(BASE_DIR.resolve())),
        "prepared_sha256": source_hash(output),
        "processing_note": entry["processing"],
        "source_version": source["revision"],
        "access_rights": source["rights"],
        "status": "Prepared; quality review pending",
        "source_url": source["url"],
        "retrieved_utc": datetime.fromtimestamp(
            source_file.stat().st_mtime, timezone.utc).isoformat(),
        "cpu_coverage": source["cpu_coverage"],
        "record_count": len(prepared["records"]),
        "page_range": source["page_range"],
        "skipped_pages": prepared["skipped"],
        "known_limitations": KNOWN_LIMITATIONS[source["locator_type"]],
    })

# Reuse the common manifest writer from Cell 4
manifest = save_manifest(lineage_records)

print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print(
    "Saved to:",
    (BASE_DIR / "data" / "week3_manifest.json").relative_to(BASE_DIR)
)

Saved lineage: data/week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 9
Saved to: data/week3_manifest.json


In [15]:
# CELL 6B — VERIFY COMMON LINEAGE MANIFEST (handout, unchanged)
# REQUIRED: Run after Cell 6A
import json

manifest_path = BASE_DIR / "data" / "week3_manifest.json"

assert manifest_path.is_file(), (
    "Common lineage manifest not found."
)

with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)

saved_records = saved_manifest.get("records", [])

assert saved_records, (
    "No lineage records found in saved manifest."
)

assert len(saved_records) == len(lineage_entries), (
    "Saved lineage count does not match registration."
)

for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]

    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"

    assert source_hash(source) == record["source_sha256"], (
        f"Source checksum mismatch: {source}"
    )

    assert source_hash(output) == record["prepared_sha256"], (
        f"Prepared output checksum mismatch: {output}"
    )

print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path.relative_to(BASE_DIR))

COMMON LINEAGE VALIDATION: PASS
Verified records: 9
Manifest: data/week3_manifest.json


## Part C: V1 data pipeline

Cell 7 is the handout's common runner with two changes (marked). Cell 8A runs the same Part B functions on all 9 sources, so Parts B and C cannot drift apart; Cell 9 checks that their outputs are byte-identical.

In [16]:
# Cell 7 — REQUIRED: Common V1 Pipeline Utilities
# Changes from handout: (1) BASE_DIR and RAW_DIR come from Cell 1 instead of
# being redefined here; (2) a "documents" folder input expands to the
# registry's source files; (3) the manifest stores paths relative to BASE_DIR.
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

# Step 1: Define standard project paths (BASE_DIR, RAW_DIR from Cell 1)
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"
PIPE_DIR.mkdir(parents=True, exist_ok=True)

# Store registered pipeline runs for repeatability verification
PIPELINE_RUNS = {}


# Step 2: Calculate SHA-256 checksum
def sha256_file(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()


# Step 3: Resolve an explicitly selected source
def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path

    if folder:
        if not source.is_dir():
            raise FileNotFoundError(
                f"Source folder not found: {source}"
            )
    else:
        if not source.is_file():
            raise FileNotFoundError(
                f"Source file not found: {source}"
            )

    return source


# Step 4: Execute the selected V1 data pipeline
def execute_pipeline(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    source = Path(source)

    # INGEST: Identify relevant source files
    if kind == "images":
        inputs = sorted(
            p for p in source.iterdir()
            if p.is_file()
            and p.suffix.lower() in {
                ".png", ".jpg", ".jpeg", ".webp"
            }
        )
    elif kind == "documents" and source.is_dir():   # added: registry-listed documents
        inputs = [source / s["relative_path"] for s in configuration["sources"]]
    else:
        inputs = [source]

    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError(
            "No valid source files to ingest."
        )

    # INSPECT: Apply data-type-specific inspection
    inspection = inspect_fn(inputs, configuration)

    # Define output directory
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    # PREPARE / STORE: Apply selected transformation
    outputs, mappings, method = prepare_fn(
        inputs,
        output_dir,
        configuration
    )

    outputs = [Path(p) for p in outputs]

    # VALIDATE: Check prepared outputs
    if not outputs:
        raise ValueError(
            "No prepared outputs were produced."
        )

    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(
                f"Prepared output missing: {output}"
            )

        if output.stat().st_size == 0:
            raise ValueError(
                f"Prepared output is empty: {output}"
            )

    # Record basic validation evidence
    validation = {
        "output_count": len(outputs),
        "output_sha256": [
            sha256_file(p) for p in outputs
        ],
        "output_bytes": [
            p.stat().st_size for p in outputs
        ]
    }

    # RECORD LINEAGE: Create pipeline execution record
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [
            {
                "path": str(p.relative_to(BASE_DIR)),
                "sha256": sha256_file(p)
            }
            for p in inputs
        ],
        "outputs": [
            str(p.relative_to(BASE_DIR)) for p in outputs
        ],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Not verified",
        "access_rights": (
            "Not verified — check before sharing"
        )
    }

    # Save pipeline manifest
    manifest_path = output_dir / "pipeline_manifest.json"

    manifest_path.write_text(
        json.dumps(
            record,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print(
        f"SUCCESS: {kind} | "
        f"Inputs: {len(inputs)} | "
        f"Outputs: {len(outputs)}"
    )

    print("Manifest:", manifest_path.relative_to(BASE_DIR))

    return record


# Step 5: Register and run a selected pathway
def register_and_run(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    # Keep the runner for repeatability verification
    def runner():
        return execute_pipeline(
            kind,
            source,
            inspect_fn,
            prepare_fn,
            configuration
        )

    first = runner()

    PIPELINE_RUNS[kind] = {
        "first": first,
        "runner": runner
    }

    return first


print("Cell 7 ready.")
print("Run ONLY the selected Cell 8 pathway(s).")
print("Common pipeline utilities are available.")

Cell 7 ready.
Run ONLY the selected Cell 8 pathway(s).
Common pipeline utilities are available.


In [17]:
# Cell 8A — CONDITIONAL REQUIRED: PDF/TXT Projects
# Changes from handout: the source is the registry folder (all 9 sources),
# and inspection/preparation call the Part B functions with CONFIG_A values.
# Pipeline-level source_version/access_rights stay as in the handout; the
# per-source values are in week3_manifest.json (Cell 6A).

# Step 1: Select the source and preparation configuration
SOURCE_A = resolve_source(".", folder=True)

CONFIG_A = {
    "method": "page-aware-lines",
    "chunk_size": CHUNK_SIZE,
    "overlap": CHUNK_OVERLAP,
    "min_page_letters": MIN_PAGE_LETTERS,
    "pypdf_version": pypdf.__version__,
    "sources": SOURCES,
}


# Step 2: Inspect document content (same functions as A3)
def inspect_documents(inputs, cfg):
    return [inspect_pdf(source) if source["locator_type"] == "pdf_page" else inspect_amd(source)
            for source in cfg["sources"]]


# Step 3: Prepare documents (same functions as Part B)
def prepare_documents(inputs, out, cfg):
    outputs, mapping = [], []
    for source_file, source in zip(inputs, cfg["sources"]):
        records, skipped = build_records(source, cfg["chunk_size"], cfg["overlap"])
        if not records:
            raise ValueError(f"No records produced for {source['source_id']}")
        target = out / f"prepared_text_{source['source_id']}.jsonl"
        write_jsonl(records, target)
        outputs.append(target)
        mapping.append({
            "source_id": source["source_id"],
            "source": str(source_file.relative_to(BASE_DIR)),
            "output": str(target.relative_to(BASE_DIR)),
            "records": len(records),
            "skipped_pages": skipped,
        })
    method = (f"{cfg['method']} chunks of at most {cfg['chunk_size']} characters, "
              f"{cfg['overlap']} overlap (manuals); {AMD_PROCESSING} (AMD pages)")
    return outputs, mapping, method


# Step 4: Execute and register the pipeline
run_A = register_and_run(
    "documents",
    SOURCE_A,
    inspect_documents,
    prepare_documents,
    CONFIG_A
)

print("Cell 8A completed successfully.")

SUCCESS: documents | Inputs: 9 | Outputs: 9
Manifest: data/pipeline_v1/documents/pipeline_manifest.json
Cell 8A completed successfully.


In [18]:
# Cell 9 — REQUIRED: Repeatability and Provenance Verification (handout)
# Added at the end: Part C outputs must be byte-identical to Part B outputs.

# Step 1: Ensure at least one pipeline has been registered
assert PIPELINE_RUNS, (
    "No pipeline runs registered. "
    "Run at least one selected Cell 8 pathway first."
)

print("--- Repeatability Verification ---")

# Step 2: Rerun each selected pipeline
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]

    # Execute the same pipeline again
    second = entry["runner"]()

    # Step 3: Verify input provenance
    assert first["sources"] == second["sources"], (
        f"Source fingerprint changed for {kind}"
    )

    # Step 4: Verify prepared output fingerprints
    assert first["validation"] == second["validation"], (
        f"Prepared output changed for {kind}"
    )

    # Step 5: Verify source-to-output mapping
    assert (
        first["source_output_mapping"]
        == second["source_output_mapping"]
    ), (
        f"Source-to-output mapping changed for {kind}"
    )

    print(
        f"PASS: {kind} — source fingerprints, "
        "prepared outputs and mappings match"
    )

# Step 6: Display final verification message
print("\nAll selected pipelines passed repeatability smoke checks.")

# Added: Part B and Part C used the same method
for source in SOURCES:
    name = f"prepared_text_{source['source_id']}.jsonl"
    assert sha256_file(PROCESSED_DIR / name) == sha256_file(PIPE_DIR / "documents" / name), (
        f"Part B and Part C outputs differ for {source['source_id']}"
    )
print("PASS: all", len(SOURCES), "Part C outputs are byte-identical to Part B.")

--- Repeatability Verification ---
SUCCESS: documents | Inputs: 9 | Outputs: 9
Manifest: data/pipeline_v1/documents/pipeline_manifest.json
PASS: documents — source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.
PASS: all 9 Part C outputs are byte-identical to Part B.
